# 2. Príznaky snímok pomocou ResNet18

**ResNet18** je konvolučná sieť predtrénovaná na ImageNet (1,2 mil. obrázkov, 1000 tried).
Použijeme ju na prevod snímok na príznaky (**transfer learning**):
- poslednú vrstvu `fc` (1000 tried ImageNet) nahradíme `nn.Identity()` – výstupom
  je vektor **512 príznakov** na snímku,
- sieť sa netrénuje, iba spracuje snímky,
- snímky normalizujeme rovnako ako pri trénovaní na ImageNet (`to_imagenet_tensor`).

Výsledok: `features_resnet.npy` s tvarom `(počet klipov, 16, 512)`.
Používajú ho notebooky 4 (CNN-SVM) a 5 (CNN-LSTM).

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

from har.config import INDEX_FILE
from har.data import read_frames, to_imagenet_tensor

df = pd.read_csv(INDEX_FILE)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval().to(device);

In [3]:
feats = []
with torch.no_grad():
    for path in df.path:
        x = to_imagenet_tensor(read_frames(path)).to(device)   # (16, 3, 224, 224)
        feats.append(model(x).cpu().numpy())                   # (16, 512)

feats = np.stack(feats)
np.save("features_resnet.npy", feats)
print(feats.shape)

(238, 16, 512)
